# TTK4115 Helicopter Lab — Part I: Post-processing of experimental data

This notebook is an initial post-processing notebook for **Part I — Monovariable control**. It is deliberately structured according to the lab assignment tasks and initially focuses on plotting the **raw logged signals without filtering or trimming**.

The signal order is based on the logging formats documented during the lab:

**Standard 7-row files**

0. Time [s]
1. Travel [rad]
2. Travel rate [rad/s]
3. Pitch [rad]
4. Pitch rate [rad/s]
5. Elevation [rad]
6. Elevation rate [rad/s]

**Parameter-identification 8-row file**

0. Time [s]
1. Elevation rate [rad/s]
2. Pitch rate [rad/s]
3. Elevation [rad]
4. Travel rate [rad/s]
5. Travel [rad]
6. Pitch [rad]
7. $V_s$ [V]

The notebook is intended to evolve into the final report post-processing notebook later.

## 0. Setup and helper functions

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import loadmat

# Change this if the notebook is moved relative to the data files.
DATA_DIR = Path(".")

# All exported report figures are placed here.
EXPORT_DIR = Path("exported_figures")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# Export formats can be changed later, e.g. ("pdf",) for LaTeX/report use.
EXPORT_FORMATS = ("png", "pdf")
EXPORT_DPI = 200

STANDARD_SIGNALS = [
    ("Travel", "rad"),
    ("Travel rate", "rad/s"),
    ("Pitch", "rad"),
    ("Pitch rate", "rad/s"),
    ("Elevation", "rad"),
    ("Elevation rate", "rad/s"),
]

KF_SIGNALS = [
    ("Elevation rate", "rad/s"),
    ("Pitch rate", "rad/s"),
    ("Elevation", "rad"),
    ("Travel rate", "rad/s"),
    ("Travel", "rad"),
    ("Pitch", "rad"),
    ("Vs", "V"),
]

def _slug(text):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", text).strip("_")

def load_logged_mat(filename):
    """Load a lab .mat file and return the numeric logged array."""
    path = DATA_DIR / filename
    content = loadmat(path)
    keys = [k for k in content if not k.startswith("__")]
    if len(keys) != 1:
        raise ValueError(f"Expected one logged variable in {filename}, found {keys}")
    data = np.asarray(content[keys[0]], dtype=float)
    if data.ndim != 2:
        raise ValueError(f"Expected a 2-D array, got shape {data.shape}")
    return data

def identify_format(data):
    """Return the documented signal layout from the number of logged rows."""
    if data.shape[0] == 7:
        return "standard", STANDARD_SIGNALS
    if data.shape[0] == 8:
        return "parameter identification", KF_SIGNALS
    return "unknown", [(f"Channel {i}", "-") for i in range(1, data.shape[0])]

def inspect_file(filename):
    data = load_logged_mat(filename)
    kind, signals = identify_format(data)
    t = data[0]
    dt = np.median(np.diff(t)) if len(t) > 1 else np.nan
    return {
        "file": filename,
        "rows": data.shape[0],
        "samples": data.shape[1],
        "format": kind,
        "duration [s]": t[-1] - t[0] if len(t) else np.nan,
        "dt [s]": dt,
    }

def plot_signal(filename, signal_index, export=True, xlim=None):
    """Plot one stored signal in its own figure and optionally export it."""
    data = load_logged_mat(filename)
    kind, signals = identify_format(data)
    if signal_index < 1 or signal_index >= data.shape[0]:
        raise IndexError(f"Signal index {signal_index} not present in {filename}")

    name, unit = signals[signal_index - 1]
    t = data[0]
    y = data[signal_index]

    fig, ax = plt.subplots(figsize=(8.0, 4.2))
    ax.plot(t, y, linewidth=1.0)
    ax.set_xlabel("Time [s]")
    ax.set_ylabel(f"{name} [{unit}]" if unit != "-" else name)
    ax.set_title(f"{filename} — {name}")
    ax.grid(True, alpha=0.3)
    if xlim is not None:
        ax.set_xlim(*xlim)
    fig.tight_layout()

    if export:
        file_dir = EXPORT_DIR / _slug(Path(filename).stem)
        file_dir.mkdir(parents=True, exist_ok=True)
        base = file_dir / f"{signal_index:02d}_{_slug(name)}"
        for ext in EXPORT_FORMATS:
            fig.savefig(base.with_suffix(f".{ext}"), dpi=EXPORT_DPI, bbox_inches="tight")
    plt.show()

def plot_all_signals(filename, export=True, xlim=None):
    """Plot every stored non-time signal as a separate figure."""
    data = load_logged_mat(filename)
    for i in range(1, data.shape[0]):
        plot_signal(filename, i, export=export, xlim=xlim)


### 0.1 Data inventory

In [ ]:
MAT_FILES = sorted(p.name for p in DATA_DIR.glob("*.mat"))
inventory = pd.DataFrame(inspect_file(name) for name in MAT_FILES)
inventory

## 1. Task 1 — Manual control

The lab assignment starts with manual feed-forward control using the joystick. No `.mat` file in the current folder can be identified unambiguously as a dedicated manual-control recording from its filename alone.

This section is therefore kept as a placeholder. If a manual-control recording is identified later, it should be added here rather than mixed with the PD-controller experiments.

## 2. Task 2 — Parameter identification

`kfestimation.mat` uses the documented 8-row logging format and contains $V_s$ in addition to the helicopter states. The plots below initially show every stored raw signal. Later this section can be extended with the selected equilibrium interval, the estimate of $V_{s,0}$, and the calculation of $K_f$.

### 2.1 `kfestimation.mat` — all raw signals

In [ ]:
plot_all_signals("kfestimation.mat")

## 3. Task 3 — PD control and pole-placement experiments

The following subsections contain all available Task 3 recordings. Pole labels below are based on the filenames where they are sufficiently explicit; they can be corrected once the experimental notes are checked.

The plots are intentionally raw at this stage. No low-pass filtering, time cropping, offset correction or reference reconstruction is applied.

### 3.1 `task3-1-1.mat`

Filename suggests poles $(-1,-1)$.

In [ ]:
plot_all_signals("task3-1-1.mat")

### 3.2 `task3-3j.mat`

The exact pole pair is not yet confirmed from the available files.

In [ ]:
plot_all_signals("task3-3j.mat")

### 3.3 `taskpocomneg1j2.mat`

Filename suggests complex poles $-1\pm 2j$.

In [ ]:
plot_all_signals("taskpocomneg1j2.mat")

### 3.4 `taskpocomneg1j10.mat`

Filename suggests complex poles $-1\pm 10j$; this also matches the current `init_heli_3_10.m`.

In [ ]:
plot_all_signals("taskpocomneg1j10.mat")

### 3.5 `taskpos11.mat`

Filename suggests poles $(+1,+1)$.

In [ ]:
plot_all_signals("taskpos11.mat")

### 3.6 `taskpos22.mat`

Filename suggests poles $(+2,+2)$.

In [ ]:
plot_all_signals("taskpos22.mat")

### 3.7 `taskposneg10.mat`

The autosave initialization file contains $\lambda_1=-1$, $\lambda_2=0$, consistent with this filename.

In [ ]:
plot_all_signals("taskposneg10.mat")

### 3.8 `task3-5-5.mat`

Filename suggests a $(-5,-5)$ test, but this recording contains only time and one constant zero channel. It is retained here for completeness.

In [ ]:
plot_all_signals("task3-5-5.mat")

## 4. Task 4 — Material for oral evaluation / report selection

This section will later contain only the **selected representative experiments and report-ready comparison figures**. Keeping this separate from the raw plots makes it possible to preserve the full data inspection above while producing a compact set of figures for the report and oral evaluation.

Planned additions after the raw-data review:

- selected stable / slow response,
- selected oscillatory or lightly damped response,
- selected aggressive / unstable response,
- comparison of representative pitch responses,
- selected parameter-identification figure(s),
- quantitative metrics only where the logged signals support them.

## 5. Notes for later processing

At this stage the notebook deliberately does **not** infer the joystick pitch reference $p_c$ or the differential motor voltage $V_d$, because these signals are not present in the standard 7-row recordings. Any later response metrics must therefore be chosen with this limitation in mind.

The next processing stage should identify useful time intervals, confirm the pole pair for each run, and decide which raw signals are actually needed in the report.